# Day 070 — Exercise 4: Variations and Grid

**What you'll build:** `generate_variations` (same prompt, N different seeds) and `create_image_grid` (stitch images into a rectangular grid).

**Why it matters:** These two functions form the exploration loop in every image generation workflow: generate N → view grid → pick best → refine prompt.

In [ ]:
import math
from PIL import Image
_mock_gen = lambda prompt, **kw: Image.new('RGB', (kw.get('width', 512), kw.get('height', 512)), 'steelblue')

def generate_image(prompt, negative='', generate_fn=None,
                   width=512, height=512, steps=20,
                   guidance_scale=7.5, seed=42):
    if generate_fn is not None:
        return generate_fn(prompt, negative=negative, width=width,
                           height=height, steps=steps,
                           guidance_scale=guidance_scale, seed=seed)
    raise RuntimeError('diffusers not available in exercises')


## Task

**`generate_variations(base_prompt, n_variations, generate_fn=None, **kwargs)`:**
- `for i in range(n_variations): seed = i * 1000 + 42`
- Call `generate_image(base_prompt, generate_fn=generate_fn, seed=seed, **kwargs)`
- Return the list of images

**`create_image_grid(images, cols=2)`:**
- Raise `ValueError` if `images` is empty
- `cols = max(1, min(cols, len(images)))`
- `rows = math.ceil(len(images) / cols)`
- `w, h = images[0].size`; create `Image.new('RGB', (cols*w, rows*h), 'white')`
- Paste each image at `(col*w, row*h)` where `col = i % cols`, `row = i // cols`

## Your Implementation

In [ ]:
def generate_variations(base_prompt: str, n_variations: int,
                         generate_fn=None, **kwargs) -> list:
    """Generate n_variations images of the same prompt with different seeds.

    Uses seeds: 42, 1042, 2042, ... (i * 1000 + 42)
    Returns:
        list of PIL.Image.Image, length == n_variations
    """
    raise NotImplementedError


def create_image_grid(images: list, cols: int = 2):
    """Arrange images in a rectangular grid.

    Args:
        images: list of PIL.Image.Image
        cols:   number of columns (capped to len(images))
    Returns:
        Single PIL.Image.Image
    Raises:
        ValueError if images is empty
    """
    raise NotImplementedError


In [ ]:
def generate_variations(base_prompt: str, n_variations: int,
                         generate_fn=None, **kwargs) -> list:
    images = []
    for i in range(n_variations):
        seed = i * 1000 + 42
        img = generate_image(base_prompt, generate_fn=generate_fn,
                             seed=seed, **kwargs)
        images.append(img)
    return images


def create_image_grid(images: list, cols: int = 2):
    if not images:
        raise ValueError('images list must not be empty')
    cols = max(1, min(cols, len(images)))
    rows = math.ceil(len(images) / cols)
    w, h = images[0].size
    grid = Image.new('RGB', (cols * w, rows * h), 'white')
    for i, img in enumerate(images):
        col = i % cols
        row = i // cols
        if img.size != (w, h):
            img = img.resize((w, h), Image.Resampling.LANCZOS)
        grid.paste(img, (col * w, row * h))
    return grid


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # generate_variations returns correct count
    imgs = generate_variations('a cat', 4, generate_fn=_mock_gen,
                               width=64, height=64)
    assert len(imgs) == 4, f"Expected 4, got {len(imgs)}"
    score += 1; print("\u2705 generate_variations returns 4 images")

    # All are PIL Images of correct size
    assert all(isinstance(i, Image.Image) and i.size == (64, 64) for i in imgs)
    score += 1; print("\u2705 all images are PIL Images with correct size")

    # Deterministic seeds: second call produces same first image
    imgs2 = generate_variations('a cat', 1, generate_fn=_mock_gen, width=64, height=64)
    assert list(imgs[0].getdata()) == list(imgs2[0].getdata())
    score += 1; print("\u2705 generate_variations is deterministic")

    # create_image_grid: 4 images, 2 cols → 2x2 grid
    tiles = [Image.new('RGB', (32, 32), c) for c in [(255,0,0),(0,255,0),(0,0,255),(255,255,0)]]
    grid = create_image_grid(tiles, cols=2)
    assert grid.size == (64, 64), f"Expected (64,64), got {grid.size}"
    score += 1; print("\u2705 2x2 grid has correct dimensions")

    # create_image_grid: empty list raises ValueError
    raised = False
    try:
        create_image_grid([])
    except ValueError:
        raised = True
    assert raised
    score += 1; print("\u2705 create_image_grid raises ValueError for empty list")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def generate_variations(base_prompt: str, n_variations: int,
                         generate_fn=None, **kwargs) -> list:
    images = []
    for i in range(n_variations):
        seed = i * 1000 + 42
        img = generate_image(base_prompt, generate_fn=generate_fn,
                             seed=seed, **kwargs)
        images.append(img)
    return images


def create_image_grid(images: list, cols: int = 2):
    if not images:
        raise ValueError('images list must not be empty')
    cols = max(1, min(cols, len(images)))
    rows = math.ceil(len(images) / cols)
    w, h = images[0].size
    grid = Image.new('RGB', (cols * w, rows * h), 'white')
    for i, img in enumerate(images):
        col = i % cols
        row = i // cols
        if img.size != (w, h):
            img = img.resize((w, h), Image.Resampling.LANCZOS)
        grid.paste(img, (col * w, row * h))
    return grid
```

**Why `i * 1000 + 42` for seeds?** Seeds 0, 1, 2 are often visually similar because the initial noise tensors are correlated. Seeds spaced 1000 apart are reliably distinct, giving meaningfully different compositions for each variation.

</details>